In [1]:
from transformers import CLIPModel
from transformers.models.clip.modeling_clip import CLIPEncoderLayer
import torch
from src.load_evaluate_utils import *
from src.dataset_utils import *

In [2]:
import sys
print(sys.executable)

/home/michele/anaconda3/envs/hf-cpu/bin/python


In [3]:
def tsv_merge(w1_pt, w1_ft, w2_pt, w2_ft, 
              k=2, whitening=True):
    
    task_matrix = w1_ft - w1_pt
    task_matrix_2 = w2_ft - w2_pt

    U1, S1, Vh1 = torch.linalg.svd(task_matrix, full_matrices=False)
    U2, S2, Vh2 = torch.linalg.svd(task_matrix_2, full_matrices=False)

    top_r = int(len(S1)/k)

    U1, U2 = U1[:, :top_r], U2[:, :top_r]
    S1, S2 = S1[:top_r], S2[:top_r]
    Vh1, Vh2 = Vh1[:top_r, :], Vh2[:top_r, :]

    U_list = [U1, U2]
    S_list = [S1, S2]
    Vh_list = [Vh1, Vh2]

    U_cat = torch.cat(U_list, dim=1)
    S_cat = torch.cat(S_list, dim=0)
    Vh_cat = torch.cat(Vh_list, dim=0)

    if whitening:
            # Safe SVD with fallback (WHITENING)
        try:
            Pu, _, QuT = torch.linalg.svd(U_cat, full_matrices=False)
        except torch._C._LinAlgError:
            print(f"⚠️ SVD failed for U in {layer}, fallback to QR.")
            Pu, _ = torch.linalg.qr(U_cat)
            QuT = Pu.T

        try:
            Pv, _, QvT = torch.linalg.svd(Vh_cat, full_matrices=False)
        except torch._C._LinAlgError:
            print(f"⚠️ SVD failed for Vh in {layer}, fallback to QR.")
            Pv, _ = torch.linalg.qr(Vh_cat)
            QvT = Pv.T

        U_orth = Pu @ QuT
        V_orth = Pv @ QvT

        M = (U_orth * S_cat) @ V_orth

    else:
        
        M = (U_cat * S_cat) @ Vh_cat
    
    w_new = (w1_pt+w2_pt)/2 + M

    return w_new


def ft_average(w1_ft, w2_ft):
    return (w1_ft + w2_ft) / 2

In [4]:
def merge_blocks(block1_ft, block2_ft, merged_block, 
                 block1_pt, block2_pt,
                 merge_matrix, merge_vector):

    for (name, w_merged) in merged_block.named_parameters():
        
        w1_ft = dict(block1_ft.named_parameters())[name]
        w2_ft = dict(block2_ft.named_parameters())[name]
        w1_pt = dict(block1_pt.named_parameters())[name]
        w2_pt = dict(block2_pt.named_parameters())[name]

        if w1_ft.dim() == 2:
            w_merged.data.copy_(merge_matrix(w1_pt, w1_ft, w2_pt, w2_ft))

        elif w1_ft.dim() == 1 and 'layer_norm' in name:
            w_merged.data.copy_(w2_ft)

        elif w1_ft.dim() == 1:
            w_merged.data.copy_(merge_vector(w1_ft, w2_ft))



In [8]:
from transformers.models.clip.modeling_clip import CLIPEncoderLayer

def merge_clip_blocks(ft_model, layer_idx, op_matrix, op_vector):

    pt_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    pt_encoder = pt_model.vision_model.encoder
    pt_layers = pt_encoder.layers

    ft_encoder = ft_model.vision_model.encoder
    ft_layers = ft_encoder.layers
    config = ft_model.config.vision_config

    layer_idx_to_delete = []

    for idx in layer_idx:

        pt_layer1 = pt_layers[idx]
        pt_layer2 = pt_layers[idx + 1]

        ft_layer1 = ft_layers[idx]
        ft_layer2 = ft_layers[idx + 1]

        merged_layer = CLIPEncoderLayer(config)

        merge_blocks(ft_layer1, ft_layer2, merged_layer, pt_layer1, pt_layer2, 
                    merge_matrix=tsv_merge, merge_vector=ft_average)

        # Replace the merged layer in the fine-tuned model and mark the second layer for deletion
        ft_layers[idx] = merged_layer
        layer_idx_to_delete.append(idx + 1)

    # Delete the second layer of each merged pair
    for idx in reversed(layer_idx_to_delete):
        del ft_layers[idx]

    ft_model.config.vision_config.num_hidden_layers = len(ft_layers)

    return ft_model


In [10]:
mnist_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
mnist_vision_model = CLIPVisionModel.from_pretrained('tanganke/clip-vit-base-patch32_mnist')

mnist_model.vision_model.load_state_dict(mnist_vision_model.vision_model.state_dict())

<All keys matched successfully>

In [11]:
merge_clip_blocks(mnist_model, layer_idx=[8, 10], op_matrix=tsv_merge, op_vector=ft_average)

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [12]:
task_name = 'MNIST'
pairs_to_reduce = [(8,9), (10,11)]

In [13]:
pt_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
pt_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Evaluation part-1: manage datasets for the specified task
ds_list = [dataset_mapping[task_name]["dataset_id"]]
data_list, lab_list = load_dataset_tasks(ds_list)
data_load, lab_map, inv_lab_map = create_dataloader_from_list(data_list)
temp_list = [dataset_mapping[task_name]["template"]]

# Evaluation part-2: generate text embeddings and evaluate the merged model
text_embeds, label_per_template, template_labels, total_num_templates = generate_text_embeddings(lab_list, temp_list, 
                                                                                                processor=pt_processor, 
                                                                                                model=pt_model,
                                                                                                device='cpu')

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Unified label mapping (sample): {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9}


In [14]:
acc = evaluate_model(mnist_model, data_load, 
                     pt_processor, text_embeds, label_per_template, lab_map, 
                     device='cpu')

print(f"Accuracy on {task_name} reducing finetuned model for {pairs_to_reduce} blocks: {acc:.5f}")

100%|██████████| 313/313 [04:08<00:00,  1.26it/s]

Accuracy on MNIST reducing finetuned model for [(8, 9), (10, 11)] blocks: 0.93900
